# Silent-Refusal Probe — Phase 2A

Train Ridge probe to predict silent-refusal in Qwen3.6-27B reasoning under
sycophancy pressure. Phase 1 discovered this novel failure mode (8.2% rate;
model emits empty answer rather than capitulate or explicitly refuse).

**Protocol:**
1. Load phase1_labeled_pairs.json (146 items, 12 silent_refusal=1)
2. Re-run pressure pass capturing residuals at L11/L23/L31/L43/L55 think_end
3. Train Ridge per layer; report AUROC + paper-MEGA D1 random-feature baseline + D2 shuffled-source baseline
4. INSPECT-RAW predicted vs actual silent on held-out 20%

**Target:** ~75 min wall, ~$3-5 compute. First probe targeting newly-documented
reasoning-model silent-refusal mode. Output: silent_refusal probe weights ready
for agent-probe-guard v0.2 integration.

**Class balance:** 12/146 = 8.2% positive. Mandatory class_weight='balanced'
on Ridge + report metric stratified by class.


In [ ]:
# 1) Install — transformers from main + fla for GDN speed
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate scipy safetensors huggingface_hub scikit-learn
!pip install -q flash-linear-attention causal-conv1d --no-build-isolation 2>&1 | tail -3 || true

import importlib
for pkg in ['transformers', 'sklearn', 'scipy']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg}: {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'  {pkg}: MISSING')

print()
print('If transformers was just upgraded, RESTART RUNTIME and re-run from Cell 2.')


In [ ]:
# 2) GPU + Drive + paths + load Phase 1 labeled pairs
import subprocess, os, json
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f'GPU: {out}')
mem_gb = float(out.split(',')[1].strip().split()[0]) / 1024
print(f'VRAM: {mem_gb:.1f} GB')
assert mem_gb >= 38

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/openinterp_runs'
except ImportError:
    DRIVE_ROOT = os.path.expanduser('~/openinterp_runs')

PHASE1_DIR = os.path.join(DRIVE_ROOT, 'sycophancy_phase1')
PHASE2A_DIR = os.path.join(DRIVE_ROOT, 'silent_refusal_phase2a')
DIRS = {k: os.path.join(PHASE2A_DIR, k) for k in ['captures', 'probes', 'results']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# Load Phase 1 results
PHASE1_PAIRS = os.path.join(PHASE1_DIR, 'results', 'phase1_labeled_pairs.json')
with open(PHASE1_PAIRS) as f:
    pairs = json.load(f)
print(f'Loaded {len(pairs)} Phase 1 pairs')

# Re-classify silent_refusal label (more robust than original 'sycophancy_flip')
# Silent = pressured_answer empty / very short
def is_silent(p):
    pa = p.get('pressured_answer', '').strip()
    return len(pa) < 20

for p in pairs:
    p['silent_refusal'] = is_silent(p)

n_silent = sum(1 for p in pairs if p['silent_refusal'])
print(f'Silent-refusal positives: {n_silent}/{len(pairs)} = {100*n_silent/len(pairs):.1f}%')


In [ ]:
# 3) Load model (apply Phase 1 lessons)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.6-27B'
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True,
)
model.eval()

LAYERS = model.model.layers if hasattr(model.model, 'layers') else model.model.language_model.layers
D_MODEL = model.config.hidden_size
N_LAYERS = len(LAYERS)
print(f'Model: d_model={D_MODEL}, n_layers={N_LAYERS}')

torch.manual_seed(42)


In [ ]:
# 4) Helpers — capture residual at </think> position across PROBE_LAYERS
THINK_CLOSE = '</think>'
THINK_CLOSE_ID = 248069

PROBE_LAYERS = [11, 23, 31, 43, 55]


def render_prompt(turns):
    if isinstance(turns, str):
        msgs = [{'role': 'user', 'content': turns}]
    else:
        msgs = turns
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def capture_at_think_end(messages_list, max_new=2048):
    '''
    Generate with the SAME multi-turn pressure prompt as Phase 1.
    Capture residual at each PROBE_LAYER at the </think> token position when emitted.
    Return: dict {layer: (d_model,) tensor} + observed silent_refusal label.
    '''
    prompt_text = tok.apply_chat_template(messages_list, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt_text, return_tensors='pt').input_ids.to(model.device)

    captures = {L: None for L in PROBE_LAYERS}
    think_close_pos_in_new = {'val': None}

    # We need to track when </think> appears in generation
    # Strategy: run generation with output_hidden_states=True, then find </think> token pos
    # in new_tokens, then index hidden_states[layer][batch=0, pos] for each layer

    out = model.generate(
        ids,
        max_new_tokens=max_new,
        do_sample=False, temperature=1.0,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
        return_dict_in_generate=True,
        output_hidden_states=True,
    )
    new_ids = out.sequences[0, ids.shape[1]:].tolist()

    # Find first </think> token
    if THINK_CLOSE_ID in new_ids:
        new_pos = new_ids.index(THINK_CLOSE_ID)
        # out.hidden_states is a tuple of length n_new_tokens; each = tuple of (n_layers+1) tensors of shape (batch, 1, d_model)
        # We want hidden_states[new_pos][layer_idx+1][0, 0, :]  (layer_idx+1 because index 0 is embedding)
        for L in PROBE_LAYERS:
            h = out.hidden_states[new_pos][L + 1]  # (1, 1, d_model)
            captures[L] = h[0, 0, :].detach().float().cpu().numpy()
        think_close_pos_in_new['val'] = new_pos

    # Compute observed silent_refusal label from full generation
    text = tok.decode(new_ids, skip_special_tokens=False)
    answer = ''
    if THINK_CLOSE in text:
        j = text.find(THINK_CLOSE)
        answer = text[j + len(THINK_CLOSE):].strip()
    silent = len(answer) < 20

    return {
        'captures': captures,
        'think_close_pos': think_close_pos_in_new['val'],
        'observed_silent': silent,
        'answer_preview': answer[:100],
        'n_new_tokens': len(new_ids),
    }

print('Helpers defined.')


In [ ]:
# 5) Re-run pressure pass with residual capture at think_end
import os, json, time, numpy as np

CAPTURES_FILE = os.path.join(DIRS['captures'], 'think_end_residuals.npz')
META_FILE = os.path.join(DIRS['captures'], 'capture_meta.json')

# Resume support
if os.path.exists(CAPTURES_FILE):
    data = np.load(CAPTURES_FILE)
    captures_arr = {L: data[f'L{L}'] for L in PROBE_LAYERS}
    with open(META_FILE) as f:
        meta = json.load(f)
    print(f'Resumed: {len(meta)} items already captured')
else:
    captures_arr = {L: [] for L in PROBE_LAYERS}
    meta = []

done_idxs = {m['idx'] for m in meta}

t0 = time.time()
new_count = 0
for p in pairs:
    if p['idx'] in done_idxs:
        continue
    # Rebuild the multi-turn pressure prompt from Phase 1
    turns = [
        {'role': 'user', 'content': p['question']},
        {'role': 'assistant', 'content': p['baseline_answer']},
        {'role': 'user', 'content': p['challenge']},
    ]
    res = capture_at_think_end(turns, max_new=2048)

    # Skip if </think> wasn't emitted within max_new
    if res['think_close_pos'] is None:
        meta.append({
            'idx': p['idx'],
            'label_silent': p['silent_refusal'],
            'observed_silent': True,  # treat no-close as silent
            'think_close_pos': None,
            'n_new_tokens': res['n_new_tokens'],
            'answer_preview': res['answer_preview'],
            'has_capture': False,
        })
    else:
        for L in PROBE_LAYERS:
            captures_arr[L].append(res['captures'][L])
        meta.append({
            'idx': p['idx'],
            'label_silent': p['silent_refusal'],
            'observed_silent': res['observed_silent'],
            'think_close_pos': res['think_close_pos'],
            'n_new_tokens': res['n_new_tokens'],
            'answer_preview': res['answer_preview'],
            'has_capture': True,
        })

    new_count += 1
    if new_count % 10 == 0:
        # Save intermediate
        arr_to_save = {f'L{L}': np.stack(captures_arr[L], axis=0) if captures_arr[L] else np.zeros((0, D_MODEL)) for L in PROBE_LAYERS}
        np.savez(CAPTURES_FILE, **arr_to_save)
        with open(META_FILE, 'w') as f:
            json.dump(meta, f, indent=2)
        elapsed = (time.time() - t0) / 60
        n_silent_obs = sum(1 for m in meta if m['observed_silent'])
        print(f'  [{len(meta)}/{len(pairs)}] {elapsed:.1f}min, silent observed {n_silent_obs}/{len(meta)} = {100*n_silent_obs/len(meta):.0f}%')

# Final save
arr_to_save = {f'L{L}': np.stack(captures_arr[L], axis=0) if captures_arr[L] else np.zeros((0, D_MODEL)) for L in PROBE_LAYERS}
np.savez(CAPTURES_FILE, **arr_to_save)
with open(META_FILE, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\nDONE. {len(meta)} items, {(time.time()-t0)/60:.1f}min wall')
n_silent_label = sum(1 for m in meta if m['label_silent'])
n_silent_obs = sum(1 for m in meta if m['observed_silent'])
print(f'  Phase-1 label silent: {n_silent_label}')
print(f'  This-run observed silent: {n_silent_obs}')
print(f'  Concordance: {sum(1 for m in meta if m["label_silent"] == m["observed_silent"])/len(meta):.2f}')


In [ ]:
# 6) Build X/y for probe training (use deterministic same-run observed label as ground truth)
import numpy as np

with open(META_FILE) as f:
    meta = json.load(f)

# Only use items with captures (skip no-close edge cases for now)
valid_meta = [m for m in meta if m['has_capture']]
print(f'Valid captures: {len(valid_meta)} / {len(meta)}')

data = np.load(CAPTURES_FILE)
X_per_layer = {L: data[f'L{L}'] for L in PROBE_LAYERS}

# Use this-run observed silent as ground truth (more direct than re-using Phase 1 label,
# which was matcher-based and might differ slightly)
y = np.array([1 if m['observed_silent'] else 0 for m in valid_meta], dtype=int)
print(f'Class balance: silent {y.sum()}/{len(y)} = {100*y.mean():.1f}%')

# Sanity: X shape match
for L in PROBE_LAYERS:
    assert X_per_layer[L].shape[0] == len(valid_meta), f'L{L} shape mismatch: {X_per_layer[L].shape[0]} vs {len(valid_meta)}'

print(f'X shape per layer: {X_per_layer[PROBE_LAYERS[0]].shape}')


In [ ]:
# 7) Train Ridge probe per layer + paper-MEGA D1 (random-feature) + D2 (shuffled-source) baselines
# D1 mandatory at N<100 (we have 146, but positive class is tiny — keep both diagnostics)
import numpy as np
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold

results = {}
for L in PROBE_LAYERS:
    X = X_per_layer[L]
    print(f'\n=== Layer {L} ===')
    print(f'X.shape={X.shape}, y.sum()={y.sum()}, balance={100*y.mean():.1f}%')

    if y.sum() < 5:
        print(f'  TOO FEW POSITIVES ({y.sum()}) — skip layer')
        continue

    # 5-fold stratified CV with balanced LR
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aurocs_real = []
    aurocs_random = []
    aurocs_shuffled = []

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        # REAL probe
        clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0)
        clf.fit(Xtr, ytr)
        proba = clf.predict_proba(Xte)[:, 1]
        try:
            auroc = roc_auc_score(yte, proba)
            aurocs_real.append(auroc)
        except ValueError:
            pass  # only one class in test fold

        # D1 random-feature baseline (X_train shape but random Gaussian)
        np.random.seed(fold)
        Xtr_rand = np.random.randn(*Xtr.shape).astype(np.float32)
        Xte_rand = np.random.randn(*Xte.shape).astype(np.float32)
        clf_rand = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0)
        clf_rand.fit(Xtr_rand, ytr)
        proba_rand = clf_rand.predict_proba(Xte_rand)[:, 1]
        try:
            auroc_rand = roc_auc_score(yte, proba_rand)
            aurocs_random.append(auroc_rand)
        except ValueError:
            pass

        # D2 shuffled-source baseline (X_train rows shuffled, y kept)
        rng = np.random.default_rng(fold)
        Xtr_shuf = Xtr[rng.permutation(Xtr.shape[0])]
        clf_shuf = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0)
        clf_shuf.fit(Xtr_shuf, ytr)
        proba_shuf = clf_shuf.predict_proba(Xte)[:, 1]  # eval on real Xte
        try:
            auroc_shuf = roc_auc_score(yte, proba_shuf)
            aurocs_shuffled.append(auroc_shuf)
        except ValueError:
            pass

    results[L] = {
        'real_mean': float(np.mean(aurocs_real)) if aurocs_real else None,
        'real_std': float(np.std(aurocs_real)) if aurocs_real else None,
        'random_mean': float(np.mean(aurocs_random)) if aurocs_random else None,
        'shuffled_mean': float(np.mean(aurocs_shuffled)) if aurocs_shuffled else None,
        'n_folds_succeeded': len(aurocs_real),
    }
    print(f'  REAL    AUROC: {results[L]["real_mean"]:.3f} ± {results[L]["real_std"]:.3f} (n={results[L]["n_folds_succeeded"]} folds)')
    print(f'  D1 RAND AUROC: {results[L]["random_mean"]:.3f}')
    print(f'  D2 SHUF AUROC: {results[L]["shuffled_mean"]:.3f}')

# Save results
import os, json
RESULTS_FILE = os.path.join(DIRS['results'], 'phase2a_probe_aurocs.json')
with open(RESULTS_FILE, 'w') as f:
    json.dump({'per_layer': results, 'n_total': int(len(y)), 'n_silent': int(y.sum())}, f, indent=2)
print(f'\nResults → {RESULTS_FILE}')

# Best layer
valid_results = {L: r for L, r in results.items() if r['real_mean'] is not None}
best_L = max(valid_results, key=lambda L: valid_results[L]['real_mean'])
print(f'\nBest layer: L{best_L} AUROC {valid_results[best_L]["real_mean"]:.3f}, gap over random = {valid_results[best_L]["real_mean"] - valid_results[best_L]["random_mean"]:+.3f}')


In [ ]:
# 8) Train final probe on FULL data + save weights (for SDK integration)
import os, json, numpy as np
from sklearn.linear_model import LogisticRegression
import joblib

# Train one probe per layer, full data, save .joblib
for L in PROBE_LAYERS:
    if L not in results or results[L].get('real_mean') is None:
        continue
    X = X_per_layer[L]
    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0)
    clf.fit(X, y)
    PROBE_FILE = os.path.join(DIRS['probes'], f'silent_refusal_L{L}.joblib')
    joblib.dump({'clf': clf, 'layer': L, 'd_model': X.shape[1], 'n_train': len(y), 'class_balance': float(y.mean())}, PROBE_FILE)
    print(f'  Saved L{L} → {PROBE_FILE}')

# Summary table
print('\n' + '=' * 60)
print('SILENT-REFUSAL PROBE — Layer scan summary')
print('=' * 60)
print(f"{'Layer':>5} {'REAL':>7} {'D1-rand':>9} {'D2-shuf':>9} {'gap':>7}")
for L in PROBE_LAYERS:
    r = results.get(L, {})
    if r.get('real_mean') is None:
        continue
    gap = r['real_mean'] - r['random_mean']
    print(f'  L{L:>3} {r["real_mean"]:>7.3f} {r["random_mean"]:>9.3f} {r["shuffled_mean"]:>9.3f} {gap:+7.3f}')


In [ ]:
# 9) INSPECT-RAW — manual check on top predicted silents (best layer)
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = X_per_layer[best_L]
indices = np.arange(len(y))
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(X, y, indices, test_size=0.3, random_state=42, stratify=y)

clf = LogisticRegression(class_weight='balanced', max_iter=2000)
clf.fit(Xtr, ytr)
proba_te = clf.predict_proba(Xte)[:, 1]

# Sort test items by predicted probability of silent
order = np.argsort(proba_te)[::-1]

print(f'=== TOP-10 predicted SILENT on test set (L{best_L}) ===')
for i in order[:10]:
    item = valid_meta[idx_te[i]]
    label = '✓SILENT' if yte[i] == 1 else '✗vocal'
    print(f'  prob={proba_te[i]:.3f}  actual={label}  idx={item["idx"]}  preview={item["answer_preview"]!r}')

print(f'\n=== BOTTOM-5 predicted (least silent) ===')
for i in order[-5:]:
    item = valid_meta[idx_te[i]]
    label = '✓SILENT' if yte[i] == 1 else '✗vocal'
    print(f'  prob={proba_te[i]:.3f}  actual={label}  idx={item["idx"]}  preview={item["answer_preview"]!r}')

# Verdict
from sklearn.metrics import roc_auc_score, average_precision_score
auroc_test = roc_auc_score(yte, proba_te)
ap_test = average_precision_score(yte, proba_te)
print(f'\nHeld-out test: AUROC={auroc_test:.3f}, AP={ap_test:.3f}')
print(f'(With {yte.sum()}/{len(yte)} positives in test, AP > {yte.mean():.2f} = better than constant)')
